# MALT + MC38 tumor / TME annotation (E14S / E15S)

## Reference choice

We use **GEO [GSE193342](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE193342)** (Liu *et al.*, mouse colonic mucosa in DSS colitis; PMID [37652012](https://pubmed.ncbi.nlm.nih.gov/37652012/)) as the scRNA reference for **immune and stromal** cell types. It does **not** contain MC38 tumor cells; epithelial calls from MALT are **colon epithelium–like**, not a literal tumor atlas.

## MC38 tumor cells

`annotate_mc38_tumor_compartment.py` runs **after** MALT: it scores **epithelial / CRC-like** (`Epcam`, `Krt8`, …), **proliferation**, **hypoxia**, and **fibroblast** modules, then assigns **`Tumor_epithelial`** (MC38 carcinoma-like) vs **`TME_*`** compartments. Output: `malt_out/query_malt_tumor_annot.h5ad` with `annotation_coarse`, `compartment`, `annotation_detailed`, `is_tumor_epithelial`.

## Pipeline

1. `prepare_mouse_colon_reference_gse193342.py` — reference `references/mouse_colon_gse193342_reference.h5ad`.
2. `export_mc38_query_for_malt.py` — `query_mc38_e14s_e15s_gex.h5ad`.
3. `marker_label_transfer.py` — MALT; **`--labels-only-output`** → `malt_out/query_malt_labeled.h5ad`.
4. `annotate_mc38_tumor_compartment.py` — tumor vs TME → `malt_out/query_malt_tumor_annot.h5ad`.

**Note:** PyTorch vs NumPy 2 may warn once; MALT uses a `.tolist()` workaround where needed.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(".").resolve()
print("ROOT", ROOT)

### 1. Build reference (~334 MB download + ~20 GB RAM peak for full matrix load)

Runs once; files are cached under `references/gse193342_cache/`.

In [ ]:
subprocess.check_call(
    [sys.executable, str(ROOT / "prepare_mouse_colon_reference_gse193342.py")],
    cwd=ROOT,
)

### 2. Export merged MC38 query (gene expression only)

In [ ]:
subprocess.check_call(
    [sys.executable, str(ROOT / "export_mc38_query_for_malt.py")],
    cwd=ROOT,
)

### 3. Run MALT (TME / colon-like types)

Adjust `--extra-markers` for validation (e.g. `Epcam,Krt8,Mki67,Slc2a1`).

In [ ]:
cmd = [
    sys.executable,
    str(ROOT / "marker_label_transfer.py"),
    "--reference",
    str(ROOT / "references" / "mouse_colon_gse193342_reference.h5ad"),
    "--query",
    str(ROOT / "query_mc38_e14s_e15s_gex.h5ad"),
    "--groupby",
    "cell_type",
    "--outdir",
    str(ROOT / "malt_out"),
    "--output-query",
    "query_malt_labeled.h5ad",
    "--labels-only-output",
    "--extra-markers",
    "Cd274,Pdcd1lg2,Cd3e,Cd79a,Lyz2,S100a8,Epcam,Krt8,Mki67,Slc2a1",
]
print(" ".join(cmd))
subprocess.check_call(cmd, cwd=ROOT)

### 4. MC38 tumor vs TME (after MALT)

Colon reference labels epithelial cells as `Epithelium_*`. MC38 carcinoma cells are **tumor epithelial**; we score epithelial + proliferation vs fibroblast modules and merge with MALT to set `compartment`, `annotation_coarse` (`Tumor`, `TME_Immune`, …), and `annotation_detailed` (includes hypoxia tag for tumor cells).

In [ ]:
import subprocess

subprocess.check_call(
    [
        sys.executable,
        str(ROOT / "annotate_mc38_tumor_compartment.py"),
        "--in",
        str(ROOT / "malt_out" / "query_malt_labeled.h5ad"),
        "--out",
        str(ROOT / "malt_out" / "query_malt_tumor_annot.h5ad"),
    ],
    cwd=ROOT,
)

In [ ]:
import scanpy as sc

q = sc.read_h5ad(ROOT / "malt_out" / "query_malt_tumor_annot.h5ad")
print(q.obs["annotation_coarse"].value_counts())
print(q.obs["compartment"].value_counts())
q.obs[
    ["sample", "malt_label", "annotation_coarse", "compartment", "is_tumor_epithelial"]
].head(10)